In [ ]:
!git clone https://github.com/orzmik/road-damage-clasification.git
%cd road-damage-clasification

Cloning into 'road-damage-clasification'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 71 (delta 22), reused 62 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 2.37 MiB | 35.23 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/road-damage-clasification


In [ ]:
!pip install ultralytics albumentations python-dotenv kaggle -q

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'
os.environ['KAGGLE_KEY'] = 'YOUR_KEY'

In [ ]:
!python scripts/download_dataset.py

Target directory prepared at: /content/road-damage-clasification
Authenticating with Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
Download complete. Dataset is ready.


In [ ]:
%%writefile scripts/train_baseline_yolo.py
"""Baseline YOLOv8 for road damage detection."""
import random, shutil
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path(__file__).parent.parent
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels-YOLO"
SPLIT_DIR = DATA_DIR / "yolo_split"

TRAIN_RATIO, VAL_RATIO, SEED = 0.70, 0.15, 42
EPOCHS, IMG_SIZE, BATCH = 30, 640, 16
DEVICE, MODEL = "cuda", "yolov8n.pt"
CLASS_NAMES = ["pothole", "crack", "manhole"]


def create_splits():
    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)
    for split in ("train", "val", "test"):
        (SPLIT_DIR / "images" / split).mkdir(parents=True)
        (SPLIT_DIR / "labels" / split).mkdir(parents=True)

    images = sorted(IMAGES_DIR.glob("*.jpg"))
    paired = [img for img in images if (LABELS_DIR / f"{img.stem}.txt").exists()]
    print(f"Found {len(images)} images, {len(paired)} with YOLO labels")

    random.seed(SEED)
    random.shuffle(paired)
    n = len(paired)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    splits = {
        "train": paired[:n_train],
        "val": paired[n_train:n_train + n_val],
        "test": paired[n_train + n_val:],
    }
    for split_name, files in splits.items():
        for img_path in files:
            label_path = LABELS_DIR / f"{img_path.stem}.txt"
            (SPLIT_DIR / "images" / split_name / img_path.name).symlink_to(img_path.resolve())
            (SPLIT_DIR / "labels" / split_name / label_path.name).symlink_to(label_path.resolve())
        print(f"  {split_name}: {len(files)} images")


def write_data_yaml():
    yaml_path = SPLIT_DIR / "data.yaml"
    yaml_path.write_text(
        f"path: {SPLIT_DIR.resolve()}\n"
        f"train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}\n"
    )
    return yaml_path


def main():
    create_splits()
    yaml_path = write_data_yaml()
    model = YOLO(MODEL)
    model.train(
        data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
        device=DEVICE, project="runs/baseline", name="yolov8n_rome", exist_ok=True,
    )
    metrics = model.val(data=str(yaml_path), split="test")
    print(f"\nTest mAP@0.5     : {metrics.box.map50:.4f}")
    print(f"Test mAP@0.5:0.95: {metrics.box.map:.4f}")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {metrics.box.maps[i]:.4f}")


if __name__ == "__main__":
    main()

Writing scripts/train_baseline_yolo.py


In [ ]:
!python scripts/train_baseline_yolo.py

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Found 2009 images, 2009 with YOLO labels
  train: 1406 images
  val: 301 images
  test: 302 images
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage-clasification/data/yolo_split/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok

In [ ]:
from google.colab import files
files.download('runs/detect/runs/baseline/yolov8n_rome/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r yolo_run.zip runs/detect/runs/baseline/yolov8n_rome/
from google.colab import files
files.download('yolo_run.zip')

  adding: runs/detect/runs/baseline/yolov8n_rome/ (stored 0%)
  adding: runs/detect/runs/baseline/yolov8n_rome/train_batch1760.jpg (deflated 10%)
  adding: runs/detect/runs/baseline/yolov8n_rome/labels.jpg (deflated 32%)
  adding: runs/detect/runs/baseline/yolov8n_rome/confusion_matrix.png (deflated 29%)
  adding: runs/detect/runs/baseline/yolov8n_rome/train_batch1.jpg (deflated 6%)
  adding: runs/detect/runs/baseline/yolov8n_rome/train_batch1761.jpg (deflated 10%)
  adding: runs/detect/runs/baseline/yolov8n_rome/train_batch2.jpg (deflated 4%)
  adding: runs/detect/runs/baseline/yolov8n_rome/confusion_matrix_normalized.png (deflated 25%)
  adding: runs/detect/runs/baseline/yolov8n_rome/BoxP_curve.png (deflated 8%)
  adding: runs/detect/runs/baseline/yolov8n_rome/val_batch2_pred.jpg (deflated 6%)
  adding: runs/detect/runs/baseline/yolov8n_rome/BoxPR_curve.png (deflated 12%)
  adding: runs/detect/runs/baseline/yolov8n_rome/BoxF1_curve.png (deflated 11%)
  adding: runs/detect/runs/baseli

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

trenowac yolo na nasz trening i nie coco

ile klasy?

piszac ile klas

analizowac shape output yolo ktory uzywalismy zeby potwierdzic ze nasz model
 sie ucze na 3 klasy bo nie potrzebujemy

patrzyc na RFD
